# Cinema 売上予測モデルの探索と視覚化

このノートブックでは、Cinema データセットを使った回帰モデルの性能を視覚的に評価します。

## 目的

- **回帰モデルの性能を視覚的に評価する** - 実測値 vs 予測値、残差プロット
- **特徴量の分布を理解する** - ヒストグラム、散布図
- **欠損値を処理する** - 欠損値の確認と補完
- **データの傾向を発見する** - 相関分析、外れ値検出

## 1. 環境セットアップとパッケージ読み込み

In [23]:
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.FastTree, 3.0.1"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: FSharp.Stats, 0.5.0"
#r "nuget: Deedle, 3.0.0"

open System
open System.IO
open Microsoft.ML
open Microsoft.ML.Data
open Plotly.NET
open Plotly.NET.LayoutObjects
open FSharp.Stats
open Deedle

printfn "✅ 環境セットアップ完了"

Installed Packages Deedle, 3.0.0 FSharp.Stats, 0.5.0 Microsoft.ML, 3.0.1 Microsoft.ML.FastTree, 3.0.1 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0

✅ 環境セットアップ完了


## 2. データ型定義

In [24]:
[<CLIMutable>]
type CinemaData = {
    [<LoadColumn(0)>] CinemaId: int
    [<LoadColumn(1)>] SNS1: float32
    [<LoadColumn(2)>] SNS2: float32
    [<LoadColumn(3)>] Actor: float32
    [<LoadColumn(4)>] Original: float32
    [<LoadColumn(5)>] Sales: float32
}

[<CLIMutable>]
type CinemaPrediction = {
    [<ColumnName("Score")>] PredictedSales: float32
}

## 3. データ読み込みと探索

In [25]:
let mlContext = MLContext(seed = Nullable 0)
let dataPath = "../data/cinema.csv"

let dataView =
    mlContext.Data.LoadFromTextFile<CinemaData>(
        dataPath,
        hasHeader = true,
        separatorChar = ',')

// データフレームに変換
let cinemaData =
    mlContext.Data.CreateEnumerable<CinemaData>(dataView, reuseRowObject = false)
    |> Seq.toList

printfn $"データ数: {cinemaData.Length} サンプル"
printfn $"\n最初の 5 件:"
cinemaData
|> List.take 5
|> List.iteri (fun i d ->
    printfn $"  {i+1}. ID={d.CinemaId}, SNS1={d.SNS1}, SNS2={d.SNS2}, Actor={d.Actor}, Original={d.Original}, Sales={d.Sales}"
)

データ数: 100 サンプル

最初の 5 件:
  1. ID=1375, SNS1=291, SNS2=1044, Actor=8808.994, Original=0, Sales=9731
  2. ID=1000, SNS1=363, SNS2=568, Actor=10290.709, Original=1, Sales=10210
  3. ID=1390, SNS1=158, SNS2=431, Actor=6340.3887, Original=1, Sales=8227
  4. ID=1499, SNS1=261, SNS2=578, Actor=8250.485, Original=0, Sales=9658
  5. ID=1164, SNS1=209, SNS2=683, Actor=10908.539, Original=0, Sales=9286


## 4. 欠損値の確認

In [26]:
let countMissing getValue name =
    let missing =
        cinemaData
        |> List.filter (fun d -> Single.IsNaN(getValue d))
        |> List.length
    printfn $"{name}: {missing} 件の欠損値 ({float missing / float cinemaData.Length * 100.0:F2}%%)"

printfn "\n=== 欠損値の確認 ==="
countMissing (fun d -> d.SNS1) "SNS1"
countMissing (fun d -> d.SNS2) "SNS2"
countMissing (fun d -> d.Actor) "Actor"
countMissing (fun d -> d.Original) "Original"
countMissing (fun d -> d.Sales) "Sales"


=== 欠損値の確認 ===
SNS1: 0 件の欠損値 (0.00%)
SNS2: 0 件の欠損値 (0.00%)
Actor: 0 件の欠損値 (0.00%)
Original: 0 件の欠損値 (0.00%)
Sales: 0 件の欠損値 (0.00%)


## 5. 基本統計量

In [27]:
let printStats name getValue =
    let values =
        cinemaData
        |> List.map (fun d -> float (getValue d))
        |> List.filter (fun v -> not (Double.IsNaN v))
    
    if values.Length > 0 then
        let mean = List.average values
        let std = Seq.stDev values
        let min = List.min values
        let max = List.max values
        printfn $"{name}:"
        printfn $"  平均={mean:F2}, 標準偏差={std:F2}, 最小={min:F2}, 最大={max:F2}"

printfn "\n=== 基本統計量 ==="
printStats "SNS1" (fun d -> d.SNS1)
printStats "SNS2" (fun d -> d.SNS2)
printStats "Actor" (fun d -> d.Actor)
printStats "Original" (fun d -> d.Original)
printStats "Sales" (fun d -> d.Sales)


=== 基本統計量 ===
SNS1:
  平均=367.03, 標準偏差=214.09, 最小=0.00, 最大=1000.00
SNS2:
  平均=658.51, 標準偏差=323.84, 最小=0.00, 最大=1500.00
Actor:
  平均=9719.36, 標準偏差=1742.39, 最小=0.00, 最大=12665.07
Original:
  平均=0.50, 標準偏差=0.50, 最小=0.00, 最大=1.00
Sales:
  平均=9882.43, 標準偏差=799.23, 最小=7869.00, 最大=11405.00


## 6. データ分布の視覚化

### 売上の分布

In [28]:
let salesValues =
    cinemaData
    |> List.map (fun d -> float d.Sales)
    |> List.filter (fun v -> not (Double.IsNaN v))

let salesHist =
    Chart.Histogram(salesValues, Name = "売上分布")
    |> Chart.withXAxisStyle(TitleText = "売上（万円）")
    |> Chart.withYAxisStyle(TitleText = "頻度")
    |> Chart.withTitle "Cinema 売上の分布"
    |> Chart.withSize(800, 500)

salesHist

<!-- Plotly chart will be drawn inside this DIV -->

### SNS1 vs 売上の散布図

In [29]:
let validData =
    cinemaData
    |> List.filter (fun d -> not (Single.IsNaN d.SNS1) && not (Single.IsNaN d.Sales))

let sns1Values = validData |> List.map (fun d -> float d.SNS1)
let salesFromSNS1 = validData |> List.map (fun d -> float d.Sales)

let sns1Scatter =
    Chart.Scatter(sns1Values, salesFromSNS1, mode = StyleParam.Mode.Markers, Name = "SNS1 vs 売上")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "SNS1")
    |> Chart.withYAxisStyle(TitleText = "売上（万円）")
    |> Chart.withTitle "SNS1 と売上の関係"
    |> Chart.withSize(800, 600)

sns1Scatter

<!-- Plotly chart will be drawn inside this DIV -->

### Actor vs 売上の散布図

In [30]:
let actorValues = cinemaData |> List.map (fun d -> float d.Actor)
let salesFromActor = cinemaData |> List.map (fun d -> float d.Sales)

let actorScatter =
    Chart.Scatter(actorValues, salesFromActor, mode = StyleParam.Mode.Markers, Name = "Actor vs 売上")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "Actor（メディア露出）")
    |> Chart.withYAxisStyle(TitleText = "売上（万円）")
    |> Chart.withTitle "Actor と売上の関係"
    |> Chart.withSize(800, 600)

actorScatter

<!-- Plotly chart will be drawn inside this DIV -->

## 7. モデルの訓練と評価

In [31]:
// F# 用のダウンキャストヘルパー関数
let downcastPipeline (x: IEstimator<_>) =
    match x with
    | :? IEstimator<ITransformer> as y -> y
    | _ -> failwith "downcastPipeline: IEstimator<ITransformer> が期待されます"

let trainTestSplit = mlContext.Data.TrainTestSplit(dataView, testFraction = 0.2, seed = Nullable 42)

let pipeline =
    mlContext.Transforms.CopyColumns("Label", "Sales")
    |> downcastPipeline
    |> fun estimator ->
        estimator
            .Append(mlContext.Transforms.ReplaceMissingValues("SNS1"))
            .Append(mlContext.Transforms.ReplaceMissingValues("SNS2"))
            .Append(mlContext.Transforms.Concatenate("Features", "SNS1", "SNS2", "Actor", "Original"))
            .Append(mlContext.Regression.Trainers.FastTree())

let model = pipeline.Fit(trainTestSplit.TrainSet)

// 予測
let predictions = model.Transform(trainTestSplit.TestSet)

// 評価
let metrics = mlContext.Regression.Evaluate(predictions, labelColumnName = "Label")

printfn "\n=== モデル評価結果 ==="
printfn $"R^2 (決定係数):      {metrics.RSquared:F4}"
printfn $"MAE (平均絶対誤差):  {metrics.MeanAbsoluteError:F2}"
printfn $"RMSE (二乗平均平方根誤差): {metrics.RootMeanSquaredError:F2}"


=== モデル評価結果 ===
R^2 (決定係数):      0.6422
MAE (平均絶対誤差):  391.34
RMSE (二乗平均平方根誤差): 474.77


## 8. 実測値 vs 予測値の可視化

In [32]:
let testData =
    mlContext.Data.CreateEnumerable<CinemaData>(trainTestSplit.TestSet, reuseRowObject = false)
    |> Seq.toList

let predictedData =
    mlContext.Data.CreateEnumerable<CinemaPrediction>(predictions, reuseRowObject = false)
    |> Seq.toList

let actualValues = testData |> List.map (fun d -> float d.Sales)
let predictedValues = predictedData |> List.map (fun p -> float p.PredictedSales)

// 散布図
let scatterChart =
    Chart.Scatter(actualValues, predictedValues, mode = StyleParam.Mode.Markers, Name = "予測結果")
    |> Chart.withMarkerStyle(Size = 10, Opacity = 0.6)

// 理想線 (y = x)
let minVal = min (List.min actualValues) (List.min predictedValues)
let maxVal = max (List.max actualValues) (List.max predictedValues)
let idealLine =
    Chart.Line([minVal; maxVal], [minVal; maxVal], Name = "理想線 (y=x)")
    |> Chart.withLineStyle(Dash = StyleParam.DrawingStyle.Dash, Color = Color.fromKeyword Red)

let predictionChart =
    [scatterChart; idealLine]
    |> Chart.combine
    |> Chart.withXAxisStyle(TitleText = "実測値（万円）")
    |> Chart.withYAxisStyle(TitleText = "予測値（万円）")
    |> Chart.withTitle "実測値 vs 予測値"
    |> Chart.withSize(800, 600)

predictionChart

<!-- Plotly chart will be drawn inside this DIV -->

## 9. 残差プロット

In [33]:
let residuals =
    List.zip actualValues predictedValues
    |> List.map (fun (actual, predicted) -> actual - predicted)

let residualChart =
    Chart.Scatter(predictedValues, residuals, mode = StyleParam.Mode.Markers, Name = "残差")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "予測値（万円）")
    |> Chart.withYAxisStyle(TitleText = "残差（実測値 - 予測値）")
    |> Chart.withTitle "残差プロット"
    |> Chart.withSize(800, 600)

residualChart

<!-- Plotly chart will be drawn inside this DIV -->

## まとめ

この Jupyter Notebook での探索により、以下のことが明らかになりました：

- **欠損値が存在する** - SNS1 と SNS2 に欠損値があり、平均値で補完
- **特徴量と売上の関係** - SNS や Actor の露出が売上に影響
- **モデルの性能** - R^2 値で予測精度を評価
- **残差の分布** - 予測誤差の傾向を視覚的に確認